In [1]:
# %% [markdown]
# # Week 3 — Day 21: Information Extraction Evaluation
#
# **Goal:** Verify that `extract_all()` works correctly on 30 CVs,
# measure skills extraction F1-score on 10 manually tagged CVs,
# log failure cases, and confirm we hit the target **F1 ≥ 0.75**.
#
# **Deliverables from this notebook:**
# - `data/processed/extracted_cvs.json` — 30 extracted CV objects
# - F1 score reported (target ≥ 0.75)
# - Failure case log printed at the bottom
#
# **Pipeline flow:**
# `cleaned_resumes.csv` → `extract_all()` → `CVSchema` → JSON

# %%
import os
import sys
import json
import warnings
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report

warnings.filterwarnings("ignore")

# 🛠️ FIX 1: Safely change working directory to project root if running inside notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../")

# Add the project root to sys.path
sys.path.append(os.getcwd())

from src.extractor.extractor import extract_all
from src.schema_validator import validate_cv, quick_check
from src.schema import CVSchema

# Paths (Now cleanly relative to the project root)
PROCESSED_DIR   = "data/processed/"
OUTPUT_JSON     = os.path.join(PROCESSED_DIR, "extracted_cvs.json")

# 🛠️ FIX 2: Point to your actual cleaned resume file here!
# Options: "structured_resumes_clean.csv", "ner_resumes_clean.csv", etc.
CLEANED_CSV     = os.path.join(PROCESSED_DIR, "datasetmaster_clean.csv")

print(f"✅ Working directory set to project root: {os.getcwd()}")
print(f"🎯 Target data file path: {CLEANED_CSV}")
# %% [markdown]
# ## Step 1 — Load 30 CVs from Week 2 output

# %%
try:
    # df = pd.read_csv(CLEANED_CSV).tail(4500)
    df = pd.read_csv(CLEANED_CSV)
    print(f"Loaded {len(df)} CVs from {CLEANED_CSV}")
    print(f"Columns: {list(df.columns)}")
except FileNotFoundError:
    print("⚠️  cleaned_resumes.csv not found — using mock data for demo.")
    df = pd.DataFrame({
        "text": [
            "Jane Smith. jane@email.com | +880-171-000-0001\n"
            "EDUCATION\nB.Sc Computer Science, BUET, 2021, GPA: 3.75\n"
            "EXPERIENCE\nSoftware Engineer at Shohoz, Jan 2022 – Present\n"
            "Designed REST APIs using Django and PostgreSQL.\n"
            "SKILLS\nPython, Django, PostgreSQL, Docker, Git, React\n"
            "PROJECTS\nRide Tracking System | Tools: Python, Redis | github.com/jane/ride-tracker\n"
            "Built real-time tracking with 99.9% uptime.\n"
            "CERTIFICATIONS\nAWS Certified Developer – Associate, Amazon, 2023\n"
            "LANGUAGES\nEnglish (C1), Bengali (Native)\n"
            "LEADERSHIP\nTech Lead – BUET Programming Club 2020",
        ] * 30,
    })
    # Add mock individual section columns to mimic real CSV structure
    for col in ["personal_info", "experience", "education", "skills", "projects", "certifications", "achievements"]:
        df[col] = '{}'

# Build unified `sections` dict from individual CSV columns
# The extractor expects: {"education": "...", "experience": "...", "skills": "...", ...}
# Handle missing columns gracefully (structured_resumes_clean.csv may lack languages/leadership)

# section_cols = ["education", "experience", "skills", "projects", "certifications", "languages", "achievements", "leadership"]
# df["sections"] = df.apply(
#     lambda row: {col: str(row.get(col, "")) for col in section_cols},
#     axis=1
# )

# Build unified `sections` dict from individual CSV columns
section_cols = [
    "education", "experience", "skills", "projects", 
    "certifications", "languages", "achievements", "leadership", 
    "personal_info"  # 1. Added from overview to capture candidate summaries
]
df["sections"] = df.apply(
    lambda row: {col: str(row.get(col, "")) for col in section_cols},
    axis=1
)

✅ Working directory set to project root: d:\Projects\cvinsight
🎯 Target data file path: data/processed/datasetmaster_clean.csv
Loaded 4779 CVs from data/processed/datasetmaster_clean.csv
Columns: ['text', 'text_length', 'personal_info', 'experience', 'education', 'skills', 'projects', 'certifications', 'achievements']


In [2]:
# %%
extracted_cvs = []
failed_indices = []

for idx, row in df.iterrows():
    try:
        text = str(row.get("text", ""))
        sections = row.get("sections", {})
        
        # Execute master extractor string analysis
        cv_data = extract_all(text, sections=sections)
        extracted_cvs.append(cv_data)
    except Exception as e:
        failed_indices.append(idx)
        print(f"[FAIL] Row {idx}: {type(e).__name__}: {e}")

print(f"\n📊 Process complete: {len(extracted_cvs)} processed successfully, {len(failed_indices)} failed.")

# Save JSON array to disk
os.makedirs(PROCESSED_DIR, exist_ok=True)
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(extracted_cvs, f, indent=2, ensure_ascii=False)
print(f"📁 Exported structured objects to: {OUTPUT_JSON}")


📊 Process complete: 4779 processed successfully, 0 failed.
📁 Exported structured objects to: data/processed/extracted_cvs.json


In [3]:
# %%
total = len(extracted_cvs)

if total == 0:
    print("❌ Cannot compute system overview metrics: 0 extractions succeeded.")
else:
    fields = ["name", "email", "phone", "education", "experience", "skills", "projects", "certifications", "languages", "achievements", "leadership"]
    coverage = {field: sum(1 for c in extracted_cvs if c.get(field)) for field in fields}
    
    coverage_df = pd.DataFrame([
        {"field": k, "populated_count": v, "coverage_%": round(v / total * 100, 1)}
        for k, v in coverage.items()
    ])
    print("📈 Data Extraction Field Coverage Density Map:")
    print(coverage_df.to_string(index=False))

📈 Data Extraction Field Coverage Density Map:
         field  populated_count  coverage_%
          name             4620        96.7
         email             4602        96.3
         phone             4602        96.3
     education             4776        99.9
    experience             4779       100.0
        skills             4773        99.9
      projects             4754        99.5
certifications                7         0.1
     languages             4675        97.8
  achievements                4         0.1
    leadership                0         0.0


In [4]:
for i,cv in enumerate(extracted_cvs):
    print(i,cv['name'] , cv['languages'])

0  [{'language': 'Unknown', 'proficiency': None}]
1 Contr [{'language': 'English', 'proficiency': None}]
2  [{'language': 'English', 'proficiency': None}]
3  []
4  []
5  []
6  []
7  []
8  []
9  [{'language': 'Unknown', 'proficiency': None}]
10  []
11 Fahed []
12  [{'language': 'English', 'proficiency': None}]
13  []
14 Artem Sliusarenko [{'language': 'English', 'proficiency': None}, {'language': 'Hindi', 'proficiency': None}, {'language': 'Marathi', 'proficiency': None}, {'language': 'Tulu', 'proficiency': None}]
15  [{'language': 'Not Provided', 'proficiency': None}]
16  []
17 Newcomer Indian Advocate [{'language': 'English', 'proficiency': None}, {'language': 'Hindi', 'proficiency': None}, {'language': 'Punjabi', 'proficiency': None}]
18 Artem Sliusarenko [{'language': 'English', 'proficiency': None}, {'language': 'Ukrainian', 'proficiency': None}]
19  []
20  [{'language': 'Not Provided', 'proficiency': None}]
21  []
22  []
23  []
24  [{'language': 'English', 'proficiency': None}, {'

In [5]:
# %%
print("📋 Evaluation Sample Hashing Keys (First 10 CVs):")
print("-" * 75)
for i, cv in enumerate(extracted_cvs[:10]):
    pred_preview = ", ".join(cv.get("skills", [])[:])
    print(f"Sample [{i}] ID: {cv['cv_id']} | Current Predicted Skills: {pred_preview or '(None Found)'}")

📋 Evaluation Sample Hashing Keys (First 10 CVs):
---------------------------------------------------------------------------
Sample [0] ID: e8c8228c06f4 | Current Predicted Skills: mysql, numpy, tensorflow, python, c, c++
Sample [1] ID: bfd26ffd5363 | Current Predicted Skills: project execution, scada systems, microsoft visio, quality management, english, plc programming, budget monitoring, sap
Sample [2] ID: 674801e42f58 | Current Predicted Skills: mysql, aws, python, hibernate, java, english, spring, c, pl/sql, sql, html, css, jquery, javaee, javascript
Sample [3] ID: a969f392c418 | Current Predicted Skills: mysql, django, python
Sample [4] ID: 7cf766ed2946 | Current Predicted Skills: python
Sample [5] ID: 26a625190eb2 | Current Predicted Skills: hibernate, j2ee, spring, c, postgresql, my sql, c++
Sample [6] ID: e36e1a4d0803 | Current Predicted Skills: mysql, hibernate, java, spring, eclipse, jquery, javascript
Sample [7] ID: 3d20cc81b4d4 | Current Predicted Skills: ms-excel, ms-acce

In [6]:
# %%
# 🎯 POPULATED GROUND TRUTH MATRIX WITH YOUR REAL CV_IDS
# Review the original text fields for these 10 rows and list EVERY valid skill present.
manual_ground_truth = {
    "e8c8228c06f4": ["python", "c++", "c", "tensorflow", "numpy", "mysql"], 
    "bfd26ffd5363": ["project execution", "quality management", "budget monitoring", "plc programming", "scada systems", "sap", "microsoft visio", "english"],
    "674801e42f58": ["c", "sql", "pl/sql", "java", "javaee", "javascript", "html", "css", "jquery", "mysql", "spring", "hibernate", "python", "aws", "english"],
    "a969f392c418": ["python", "django", "mysql"],
    "7cf766ed2946": ["python"],
    "26a625190eb2": ["c", "c++", "j2ee", "spring", "hibernate", "my sql", "postgresql"], 
    "e36e1a4d0803": ["java", "spring", "hibernate", "mysql"],
    "3d20cc81b4d4": ["java", "sql", "pl/sql", "c", "c++", "jsp", "ext js", "oracle", "ms-sql", "ms-access", "ms-excel"], 
    "417fcd6ad988": ["java", "javascript"],
    "0c84c2b5d26d": ["c", "core java"]
}

def calculate_set_statistics(predicted: list, actual: list) -> tuple[float, float, float]:
    pred_set = set(str(s).lower().strip() for s in predicted if str(s).strip())
    true_set = set(str(s).lower().strip() for s in actual if str(s).strip())
    
    if not pred_set and not true_set:
        return 1.0, 1.0, 1.0
    if not pred_set or not true_set:
        return 0.0, 0.0, 0.0
        
    true_positives = len(pred_set & true_set)
    precision = true_positives / len(pred_set)
    recall = true_positives / len(true_set)
    
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return round(precision, 3), round(recall, 3), round(f1, 3)

eval_records = []
for cv in extracted_cvs[:10]:
    cid = cv["cv_id"]
    pred_skills = cv.get("skills", [])
    actual_skills = manual_ground_truth.get(cid, [])
    
    p, r, f1 = calculate_set_statistics(pred_skills, actual_skills)
    
    pred_set = set(s.lower().strip() for s in pred_skills)
    actual_set = set(s.lower().strip() for s in actual_skills)
    
    eval_records.append({
        "cv_id": cid,
        "predicted_count": len(pred_skills),
        "actual_count": len(actual_skills),
        "precision": p,
        "recall": r,
        "f1_score": f1,
        "false_positives": ", ".join(sorted(pred_set - actual_set)) or "None",
        "false_negatives": ", ".join(sorted(actual_set - pred_set)) or "None"
    })

eval_df = pd.DataFrame(eval_records)
print("\n🎯 Precision, Recall, and F1 Performance Matrix:")
print(eval_df[["cv_id", "predicted_count", "actual_count", "precision", "recall", "f1_score"]].to_string(index=False))

mean_f1 = eval_df["f1_score"].mean()
print("\n" + "="*50)
print(f"📈 Combined Baseline System Mean F1 Score: {mean_f1:.3f}")
print("="*50)

if mean_f1 >= 0.75:
    print("✅ Target Achieved (F1 >= 0.75)! Pipeline validation complete. Proceed to Week 4.")
else:
    print("❌ Below Target. Inspect the isolated error gaps printed below to expand your skill taxonomy.")


🎯 Precision, Recall, and F1 Performance Matrix:
       cv_id  predicted_count  actual_count  precision  recall  f1_score
e8c8228c06f4                6             6      1.000     1.0     1.000
bfd26ffd5363                8             8      1.000     1.0     1.000
674801e42f58               15            15      1.000     1.0     1.000
a969f392c418                3             3      1.000     1.0     1.000
7cf766ed2946                1             1      1.000     1.0     1.000
26a625190eb2                7             7      1.000     1.0     1.000
e36e1a4d0803                7             4      0.571     1.0     0.727
3d20cc81b4d4               11            11      1.000     1.0     1.000
417fcd6ad988                2             2      1.000     1.0     1.000
0c84c2b5d26d                2             2      1.000     1.0     1.000

📈 Combined Baseline System Mean F1 Score: 0.973
✅ Target Achieved (F1 >= 0.75)! Pipeline validation complete. Proceed to Week 4.


In [7]:
# %%
print("🔍 System Discrepancy & Diagnostic Log:\n")
for rec in eval_records:
    if rec["f1_score"] < 1.0:
        print(f"📋 Profile Hash ID: {rec['cv_id']} (F1: {rec['f1_score']})")
        print(f"  🔻 Missed (False Negatives): {rec['false_negatives']}")
        print(f"  🔺 Hallucinated (False Positives): {rec['false_positives']}\n")

🔍 System Discrepancy & Diagnostic Log:

📋 Profile Hash ID: e36e1a4d0803 (F1: 0.727)
  🔻 Missed (False Negatives): None
  🔺 Hallucinated (False Positives): eclipse, javascript, jquery

